# LangGraph document analysis with conditional routing

An original, deterministic LangGraph companion lesson.

## Learning objectives

- Model document analysis as a typed shared state.
- Separate extraction, arithmetic-tool, summary, and review nodes.
- Route complete and incomplete documents deterministically.

## Setup

The offline graph runner uses only `re` and `typing`. It keeps the same state, node, and conditional-route boundaries as LangGraph while remaining executable without an optional framework install.


Install the optional graph and provider packages with `python -m pip install langgraph langchain-openai`. Optional tracing uses `python -m pip install langfuse`.

## Complete implementation

Run the next cell to define the typed state, nodes, routing, and complete offline runner.

In [1]:
import re
from typing import Literal, TypedDict

class AnalysisState(TypedDict):
    document_text: str
    numbers: list[int]
    total: int
    route: str
    summary: str
    status: str
    trace: list[str]

def extraction_node(state: AnalysisState) -> dict:
    """Extract whole-number amounts from the document text."""
    numbers = [int(value) for value in re.findall(r"\b\d+\b", state["document_text"])]
    return {"numbers": numbers, "trace": state["trace"] + ["extract"]}

def route_after_extraction(state: AnalysisState) -> Literal["calculate", "review"]:
    """Choose arithmetic when values exist, otherwise request review."""
    return "calculate" if state["numbers"] else "review"

def add_values(values: list[int]) -> int:
    """Deterministic arithmetic tool used by the graph node."""
    return sum(values)

def arithmetic_tool_node(state: AnalysisState) -> dict:
    total = add_values(state["numbers"])
    return {
        "total": total,
        "route": "summarize",
        "trace": state["trace"] + ["calculate"],
    }

def summary_node(state: AnalysisState) -> dict:
    summary = (
        f"The document contains {len(state['numbers'])} values "
        f"totaling {state['total']} EUR."
    )
    return {
        "summary": summary,
        "status": "complete",
        "trace": state["trace"] + ["summarize"],
    }

def review_node(state: AnalysisState) -> dict:
    return {
        "route": "review",
        "summary": "No numeric values were found; review the document manually.",
        "status": "needs_review",
        "trace": state["trace"] + ["review"],
    }

def run_document_graph(document_text: str) -> AnalysisState:
    state: AnalysisState = {
        "document_text": document_text,
        "numbers": [],
        "total": 0,
        "route": "pending",
        "summary": "",
        "status": "pending",
        "trace": [],
    }
    state.update(extraction_node(state))
    next_node = route_after_extraction(state)
    if next_node == "calculate":
        state.update(arithmetic_tool_node(state))
        state.update(summary_node(state))
    else:
        state.update(review_node(state))
    return state

analysis_state = run_document_graph(
    "The invoice contains line items of 19 EUR and 23 EUR."
)
review_state = run_document_graph("The invoice contains no listed amounts.")

## Deterministic assertions

Run these checks to verify graph behavior without credentials, network access, or third-party packages.

In [2]:
assert analysis_state == {
    "document_text": "The invoice contains line items of 19 EUR and 23 EUR.",
    "numbers": [19, 23],
    "total": 42,
    "route": "summarize",
    "summary": "The document contains 2 values totaling 42 EUR.",
    "status": "complete",
    "trace": ["extract", "calculate", "summarize"],
}
assert route_after_extraction(analysis_state) == "calculate"
assert add_values([19, 23]) == 42
assert review_state["route"] == "review"
assert review_state["status"] == "needs_review"
assert review_state["trace"] == ["extract", "review"]

print("Offline assertions passed.")

Offline assertions passed.


## Results interpretation

The state records every meaningful result, and each node returns only its update. The extraction route prevents arithmetic from running when the input has no values. For the sample invoice, the arithmetic tool produces 42 and the summary node marks the analysis complete.

### Exercise

Change the sample invoice to three line items whose sum is 42. Predict the trace and summary, then run the offline assertions with your expected values.

### Troubleshooting

If an amount is not extracted, inspect the `\b\d+\b` pattern: this lesson intentionally accepts whole numbers only. If a trace is out of order, check that each node appends to the incoming trace instead of replacing it.

## Credentials

Offline cells require none. The optional live cell checks `OPENAI_API_KEY` before importing LangGraph or `langchain-openai` and never prints the key. Langfuse tracing is attempted only when both `LANGFUSE_PUBLIC_KEY` and `LANGFUSE_SECRET_KEY` exist; their values are never displayed.

In [3]:
import os

live_model_provider = os.getenv("LIVE_MODEL_PROVIDER", "openai").strip().casefold()
openai_api_key = os.getenv("OPENAI_API_KEY")
if live_model_provider != "local" and not openai_api_key:
    print("SKIPPED: Set OPENAI_API_KEY to run this optional LangGraph example.")
else:
    try:
        from typing import Literal, TypedDict
        from langgraph.graph import END, START, StateGraph
    except ImportError:
        print("SKIPPED: LangGraph or langchain-openai is not installed; run `python -m pip install langgraph langchain-openai`.")
    else:
        try:
            if live_model_provider == "local":
                from types import SimpleNamespace
                from smolagents import TransformersModel
            else:
                from langchain_openai import ChatOpenAI
        except ImportError:
            if live_model_provider == "local":
                print("SKIPPED: smolagents is not installed; run `python -m pip install smolagents`.")
            else:
                print("SKIPPED: LangGraph or langchain-openai is not installed; run `python -m pip install langgraph langchain-openai`.")
        else:
            callbacks = []
            langfuse_public_key = os.getenv("LANGFUSE_PUBLIC_KEY")
            langfuse_secret_key = os.getenv("LANGFUSE_SECRET_KEY")
            if langfuse_public_key and langfuse_secret_key:
                try:
                    from langfuse.langchain import CallbackHandler
                except ImportError:
                    print("SKIPPED: Langfuse tracing is not installed; run `python -m pip install langfuse`.")
                else:
                    callbacks.append(CallbackHandler())
                    print("Langfuse tracing configured.")
            else:
                print("SKIPPED: Set both LANGFUSE_PUBLIC_KEY and LANGFUSE_SECRET_KEY to enable optional tracing.")

            if live_model_provider == "local":
                local_transformers_model = TransformersModel(
                    model_id="Qwen/Qwen2.5-Coder-1.5B-Instruct",
                    device_map="cpu",
                    torch_dtype="auto",
                    max_new_tokens=256,
                )

                class LocalChatAdapter:
                    def __init__(self, model):
                        self._model = model

                    def invoke(self, prompt, config=None):
                        response = self._model(
                            [
                                {
                                    "role": "user",
                                    "content": [
                                        {"type": "text", "text": str(prompt)}
                                    ],
                                }
                            ]
                        )
                        return SimpleNamespace(
                            content=str(getattr(response, "content", response))
                        )

                live_model = LocalChatAdapter(local_transformers_model)
            else:
                live_model = ChatOpenAI(model="gpt-5-mini")
            class LiveAnalysisState(TypedDict):
                document_text: str
                route: str
                live_summary: str
                status: str

            def prepare_live_document(state: LiveAnalysisState) -> dict:
                route = "summarize" if state["document_text"].strip() else "review"
                return {"route": route}

            def route_live_document(
                state: LiveAnalysisState,
            ) -> Literal["summarize", "review"]:
                return state["route"]

            def summarize_live_document(state: LiveAnalysisState) -> dict:
                response = live_model.invoke(
                    "Summarize this invoice in one sentence: " + state["document_text"],
                    config={"callbacks": callbacks},
                )
                return {"live_summary": str(response.content), "status": "complete"}

            def review_live_document(state: LiveAnalysisState) -> dict:
                return {"live_summary": "Manual review required.", "status": "needs_review"}

            live_builder = StateGraph(LiveAnalysisState)
            live_builder.add_node("prepare", prepare_live_document)
            live_builder.add_node("summarize", summarize_live_document)
            live_builder.add_node("review", review_live_document)
            live_builder.add_edge(START, "prepare")
            live_builder.add_conditional_edges(
                "prepare",
                route_live_document,
                {"summarize": "summarize", "review": "review"},
            )
            live_builder.add_edge("summarize", END)
            live_builder.add_edge("review", END)
            live_graph = live_builder.compile()
            live_analysis_state = live_graph.invoke(
                {
                    "document_text": "The invoice contains line items of 19 EUR and 23 EUR.",
                    "route": "pending",
                    "live_summary": "",
                    "status": "pending",
                }
            )
            print(live_analysis_state["live_summary"])

Langfuse tracing configured.


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

[transformers] The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


The invoice includes two line items totaling 42 EUR.


## Completion checklist

- [ ] Inspected the typed analysis state.
- [ ] Followed extraction, arithmetic, and summary updates.
- [ ] Exercised both the summarize and review routes.

## Attribution

This graph and its sample data are original. The live API shape is informed by the official [LangGraph Graph API guide](https://docs.langchain.com/oss/python/langgraph/use-graph-api) and [LangChain OpenAI integration guide](https://docs.langchain.com/oss/python/integrations/chat/openai). No upstream notebook code is copied. Notebook metadata carries the Apache-2.0 license.